# C2 — the timing model on a real T4

**Why.** Phase 6 matched every hit rate on two live T4s, but latency came out 5–222× the prediction:
decode steps ran 2.7–4.0× what the fitted profile says. Two suspects: the profile has no term for
context length (Phase 6's prompts are ~1,700 tokens), or CPU contention (two engines and a client on
4 vCPUs). This notebook separates them, and measures what a context-aware model is fitted on.

| Part | What it measures | Replica 1 |
|---|---|---|
| **A** | decode step against context length × batch size — 28 cells × 2 | idle |
| **B** | prefill time with and without a cached prefix — 15 cells × 3 | idle |
| **C** | the same decode cells alone and with replica 1 and a client busy (engine-side latency) | busy / idle |
| **D** | two held-out runs of Phase 6's traffic on one engine, which the fitted model must predict | idle |

**Registered before the run** (`configs/c2_timing.yaml`, `docs/09-hypotheses.md`): **C2a** — the
decode step at 1,536 tokens is ≥ 1.5× the step at 128 (batch 8); **C2b** — contention slows the
engine by ≥ 20%; **C2c** — the context-aware model, fitted on A and B only, predicts D's p50 and p95
TTFT within 2×. Predicted: C2a holds, C2b fails, C2c holds. This notebook decides nothing: it
measures, and `scripts/c2_analyze.py` decides.


**How to run.** Kaggle → *Settings* → Accelerator **GPU T4 ×2**, Internet **on** → *Run All*. About
an hour: ~12 minutes to install and start two servers, then parts A–D. Results are saved after every
part, so if a cell stops with ⛔, download **`c2_results.zip`** anyway — the parts before it are in
it. At the end, download `c2_results.zip` from the Output panel and bring it back.


## 0 — Configuration


In [ ]:
import json

# ── Generated from configs/c2_timing.yaml. Do not edit here: edit the config and re-run
#    scripts/c2_prepare.py, or the registration stops describing this notebook.
MODEL = "Qwen/Qwen3-4B"
VLLM_PINNED = "0.30.0"
BLOCK_SIZE = 16
NUM_GPU_BLOCKS = 1537          # --num-gpu-blocks-override; one is vLLM's null block
MAX_MODEL_LEN = 2048
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 8192
N_REPLICAS = 2
MEASURE = 0                 # every measured part runs on this replica
DESIGN = json.loads(r"""{"decode": {"output_tokens": 64, "skip_tokens": 8, "reps": 2, "grid": [[128, 1], [128, 2], [128, 4], [128, 8], [128, 16], [128, 32], [128, 64], [512, 1], [512, 2], [512, 4], [512, 8], [512, 16], [512, 32], [1024, 1], [1024, 2], [1024, 4], [1024, 8], [1024, 16], [1536, 1], [1536, 2], [1536, 4], [1536, 8], [1536, 12], [1984, 1], [1984, 2], [1984, 4], [1984, 8], [1984, 10]]}, "prefill": {"reps": 3, "grid": [[0, 64], [0, 256], [0, 512], [0, 1024], [0, 1536], [0, 1984], [256, 64], [256, 256], [256, 704], [768, 64], [768, 256], [768, 704], [1280, 64], [1280, 256], [1280, 704]]}, "cpu": {"cells": [[512, 8], [1024, 16], [1536, 8]], "reps": 2, "settle_s": 20, "background": {"replica": 1, "concurrency": 48, "shared_prefix_tokens": 1344, "private_tokens": 192, "output_tokens": 64}}}""")

# ── Box settings. Safe to change: nothing predicted depends on them.
GPU_MEM_UTIL = 0.85
PORTS = [8000, 8001]          # replica i listens on PORTS[i] and runs on GPU i
STARTUP_TIMEOUT_S = 1800      # the first start downloads ~8 GB of weights and compiles
OUT_DIR = "c2_results"


## 1 — Environment: two GPUs, one server each


In [ ]:
import hashlib
import json
import os
import pathlib
import platform
import random
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

pathlib.Path(OUT_DIR).mkdir(exist_ok=True)


class Stop(Exception):
    """Halts Run-All with a reason. Download phase6_results.zip anyway."""


def sh(cmd, timeout=60):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True,
                              timeout=timeout, check=False).stdout.strip()
    except Exception as e:  # an unfamiliar box: any binary may be missing
        return f"<failed: {e}>"


ENV = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "cpu_count": os.cpu_count(),
    "gpus": sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version "
               "--format=csv,noheader"),
    "kaggle": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
}
for k, v in ENV.items():
    print(f"{k:10s} {v}")
_n_gpus = len([ln for ln in ENV["gpus"].splitlines() if ln.strip() and "failed" not in ln])
if _n_gpus < N_REPLICAS:
    raise Stop(f"⛔ {_n_gpus} GPU(s) visible, {N_REPLICAS} needed: one server per GPU is the "
               "whole point. Kaggle: Settings → Accelerator → GPU T4 ×2.")
if "T4" not in ENV["gpus"]:
    print("\n⚠️  Not a Tesla T4. Hit rates stay valid (cache counts are hardware-independent), "
          "but every timing must be labelled with the GPU above, not as a T4 figure.")


## 2 — Install vLLM, pinned


In [ ]:
# PINNED: the predictions were made for this version, and Phase 0 v2 passed every check on it.
!pip install -q vllm==0.30.0


## 3 — Audit: does the installed engine contain the code the model cites?


In [ ]:
import importlib.metadata

import vllm

VLLM_VERSION = vllm.__version__
if VLLM_VERSION != VLLM_PINNED:
    raise Stop(f"⛔ vllm {VLLM_VERSION} is imported, but the predictions were made for "
               f"{VLLM_PINNED}. Restart the kernel and run from the top.")


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


ENV["packages"] = {p: _version(p) for p in ("vllm", "torch", "transformers", "triton")}
print("packages:", ENV["packages"])

ROOT = pathlib.Path(vllm.__file__).parent
SOURCES = {}
for p in ROOT.rglob("*.py"):
    try:
        SOURCES[str(p.relative_to(ROOT))] = p.read_text(errors="replace")
    except OSError:
        pass


def found_in(needle):
    return next((f for f, text in SOURCES.items() if needle in text), None)


# (what the simulator relies on, the exact source text in vLLM that implements it)
CITED = [
    ("null block reserved at start-up", "self.null_block = self.free_block_queue.popleft()"),
    ("uncached blocks reused first (prepend)", "prepend_n"),
    ("finished requests freed tail-first", "Free blocks in reverse order"),
    ("lookup capped at prompt_length - 1", "max_cache_hit_length = request.num_tokens - 1"),
    ("queries counted at admission only",
     "Record at admission so unscheduled lookups are not counted"),
    ("eviction event fired before hash check", "on_block_evicted(block)"),
    ("block access history bounded to 4", "deque(maxlen=4)"),
    ("reset also clears block metrics", "self.metrics_collector.reset()"),
    ("async scheduler caches on each output",
     "request, request.num_computed_tokens - request.num_output_placeholders"),
    ("full-prompt admission check, default on", "scheduler_reserve_full_isl: bool = True"),
    ("admission watermark, default 0", "watermark: float = Field(default=0.0"),
]
# (what this notebook sends or reads, the source text that serves it)
USED = [
    ("metric vllm:prefix_cache_queries", "vllm:prefix_cache_queries"),
    ("metric vllm:prefix_cache_hits", "vllm:prefix_cache_hits"),
    ("metric vllm:prompt_tokens_by_source", "vllm:prompt_tokens_by_source"),
    ("metric vllm:num_preemptions", "vllm:num_preemptions"),
    ("metric vllm:kv_block_lifetime_seconds", "vllm:kv_block_lifetime_seconds"),
    ("metric vllm:kv_block_idle_before_evict_seconds", "vllm:kv_block_idle_before_evict_seconds"),
    ("metric vllm:kv_block_reuse_gap_seconds", "vllm:kv_block_reuse_gap_seconds"),
    ("metric vllm:num_requests_running", "vllm:num_requests_running"),
    ("flag --num-gpu-blocks-override", "num_gpu_blocks_override"),
    ("flag --enable-prompt-tokens-details", "enable_prompt_tokens_details"),
    ("request field ignore_eos", "ignore_eos"),
    ("POST /reset_prefix_cache", '"/reset_prefix_cache"'),
    ("GET /server_info", '"/server_info"'),
]
AUDIT = {}
print(f"\nvllm {VLLM_VERSION}: {len(SOURCES):,} source files read")
for group, rows in (("the simulator cites", CITED), ("this notebook uses", USED)):
    print(f"── {group}")
    for what, needle in rows:
        AUDIT[what] = found_in(needle)
        print(f"  {'OK' if AUDIT[what] else '--'}  {what:44s} {AUDIT[what] or 'NOT FOUND'}")
AUDIT["vllm:prefix_cache_hit_rate (must NOT exist)"] = found_in('"vllm:prefix_cache_hit_rate"')
print(f"  {'OK' if not AUDIT['vllm:prefix_cache_hit_rate (must NOT exist)'] else '!!'}  "
      "vllm:prefix_cache_hit_rate does not exist (ADR-008)")

missing_used = [w for w, _ in USED if AUDIT[w] is None]
missing_cited = [w for w, _ in CITED if AUDIT[w] is None]
if missing_used:
    raise Stop(f"⛔ this vLLM lacks what the notebook needs: {missing_used}")
if missing_cited:
    print(f"\n⚠️  not found verbatim: {missing_cited}. Run anyway — the counts will show whether "
          "the behaviour itself changed.")


## 4 — Part D's held-out traffic (embedded — do not edit)


In [ ]:
# ── The traffic, regenerated — a port of kv_routing_sim/traces/generator.py (AgenticWorkload) and
#    traces/hashing.py (chain_hashes). The fingerprints prove it made the exact ids predicted on.
WORKLOADS = json.loads(r"""{"K1_half": {"n_sessions": 120, "turns_per_session": 4, "session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "seed": 0, "prefix_sharing_ratio": 0.8, "arrival_rate": 0.5}, "K64_quarter": {"n_sessions": 120, "turns_per_session": 4, "session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "seed": 0, "prefix_sharing_ratio": 0.8, "arrival_rate": 0.25, "n_shared_prefixes": 64, "prefix_zipf_s": 1.0}}""")
FINGERPRINTS = json.loads(r"""{"K1_half": "66b4574a0067fb61fc9903e2da01fddbb9a52eaebb5431453a35244110024428", "K64_quarter": "aeffe86955c67b30ecbab18afa0c30942ff617a149ee1cd3ce7cf43860a96136"}""")
PRED = json.loads(r"""{"runs": {"K1_half": {"context-free profile": {"hit_rate": 0.923039, "preemptions": 0, "prefix_cache_hits": 753200, "prefix_cache_queries": 816000, "ttft_p50_ms": 192.757, "ttft_p95_ms": 325.614}}, "K64_quarter": {"context-free profile": {"hit_rate": 0.830412, "preemptions": 0, "prefix_cache_hits": 677616, "prefix_cache_queries": 816000, "ttft_p50_ms": 183.063, "ttft_p95_ms": 1198.78}}}}""")
PREDICTIONS_SHA256 = "d39442f3a2f9f2e52ca5f378316fc720a2611cd1514c6c74a417d4154b1a84a6"

_SHARED_BASE, _PREFIX_STRIDE, _SESSION_BASE, _TURN_BASE = 0, 100_000, 10_000_000, 500_000_000
_MIX_MUL, _MIX_ADD, _MASK64 = 0x9E3779B97F4A7C15, 0x632BE59BD9B4E019, 0xFFFFFFFFFFFFFFFF
_LOW, _HIGH = 1000, 100_000  # ENGINE_SAFE_IDS: ordinary text tokens in Qwen3's vocabulary


def _engine_id(raw):
    return _LOW + ((raw * _MIX_MUL + _MIX_ADD) & _MASK64) % (_HIGH - _LOW)


def generate(w):
    """AgenticWorkload._iter_with_tokens + prefix_assignment, sorted by arrival like generate()."""
    turns, ctx = w["turns_per_session"], w["session_context_tokens"]
    user_n, asst_n, r = w["user_turn_tokens"], w["assistant_turn_tokens"], w["prefix_sharing_ratio"]
    mean_private = ctx + user_n + (user_n + asst_n) * (turns - 1) / 2.0
    shared_len = 0 if r <= 0.0 else max(0, round(r / (1.0 - r) * mean_private))
    k_total = w.get("n_shared_prefixes", 1)
    prefixes = [[_SHARED_BASE + k * _PREFIX_STRIDE + i for i in range(shared_len)]
                for k in range(k_total)]
    if k_total == 1:
        assign = [0] * w["n_sessions"]
    else:
        ks = range(k_total)
        weights = [1.0 / (k + 1) ** w.get("prefix_zipf_s", 1.0) for k in ks]
        assign = random.Random(f"shared-prefix-{w['seed']}").choices(ks, weights=weights,
                                                                     k=w["n_sessions"])
    rng, clock, out = random.Random(w["seed"]), 0.0, []
    for s in range(w["n_sessions"]):
        clock += rng.expovariate(w["arrival_rate"])
        shared = prefixes[assign[s]]
        private = [_SESSION_BASE + s * 100_000 + i for i in range(ctx)]
        history = []
        for t in range(turns):
            base = _TURN_BASE + s * 100_000 + t * 1_000
            user = [base + i for i in range(user_n)]
            out.append({"request_id": f"s{s}-t{t}",
                        "arrival_ts": round(clock + t * w["think_time_s"], 6),
                        "session_id": f"s{s}",
                        "ids": [_engine_id(x) for x in shared + private + history + user]})
            history = history + user + [base + 500 + i for i in range(asst_n)]
    out.sort(key=lambda q: q["arrival_ts"])
    return out


def _h64(data):
    return int.from_bytes(hashlib.blake2b(data, digest_size=8).digest(), "big")


def chain_hashes(ids, block_size=BLOCK_SIZE, salt=0):
    """traces/hashing.py: one hash per FULL block, each chained to its parent."""
    out, parent = [], _h64(salt.to_bytes(8, "big"))
    for i in range(len(ids) // block_size):
        body = b"".join(int(t).to_bytes(8, "big", signed=True)
                        for t in ids[i * block_size:(i + 1) * block_size])
        parent = _h64(parent.to_bytes(8, "big") + body)
        out.append(parent)
    return out


def _canonical(obj):
    return json.dumps(obj, separators=(",", ":"), sort_keys=True).encode()


TRACES = {}
for _name, _w in WORKLOADS.items():
    _tr = generate(_w)
    _fp = hashlib.sha256(_canonical([[q["request_id"], q["arrival_ts"], q["session_id"], q["ids"]]
                                     for q in _tr])).hexdigest()
    if _fp != FINGERPRINTS[_name]:
        raise Stop(f"⛔ {_name}: the regenerated trace does not match its fingerprint — this "
                   "Python's random module or the generator port differs from the one predicted on")
    for _q in _tr:
        _q["blocks"] = chain_hashes(_q["ids"])
    TRACES[_name] = _tr
    print(f"{_name:4s} ✓  {len(_tr)} requests · prompts {min(len(q['ids']) for q in _tr)}–"
          f"{max(len(q['ids']) for q in _tr)} tokens · fingerprint {_fp[:16]}…")
if hashlib.sha256(_canonical(PRED)).hexdigest() != PREDICTIONS_SHA256:
    raise Stop("⛔ the embedded predictions do not match their hash: the data cell was edited")
print(f"predictions ✓ {PREDICTIONS_SHA256[:16]}…\n\nThe context-free profile's predictions (one replica), reported:")
for _name, _by in PRED["runs"].items():
    print(f"  {_name:4s} " + "   ".join(f"{p} {v['hit_rate']:.4f}" for p, v in _by.items()))


## 5 — Two servers


In [ ]:
SERVERS = {}  # replica -> {"proc", "logf", "log_path"}
RESET_MODE = "endpoint"  # falls back to "restart" if the dev endpoint is not served


def engine_argv(i):
    return [
        "vllm", "serve", MODEL, "--port", str(PORTS[i]),
        "--dtype", "half",  # T4 has no bf16
        "--block-size", str(BLOCK_SIZE),
        "--num-gpu-blocks-override", str(NUM_GPU_BLOCKS),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--enable-prefix-caching",
        "--enable-prompt-tokens-details",
        "--seed", "0",
    ]


def port_in_use(port):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0


def start_server(i):
    # ⚠️ Refuse to launch while ANYTHING listens on the port: a stale server would answer /health
    # before ours had time to die of "address already in use" (found by Phase 3's dry-run).
    for _ in range(30):
        if not port_in_use(PORTS[i]):
            break
        time.sleep(1)
    else:
        raise Stop(f"⛔ something already serves on port {PORTS[i]} — a server from an earlier "
                   "run? Run the clean-up cell (or !pkill -f 'vllm serve') and start again.")
    # One physical GPU per replica: complete isolation, so TTFT is attributable (ADR-005).
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), VLLM_SERVER_DEV_MODE="1")
    log_path = pathlib.Path(OUT_DIR) / f"server_{i}.log"
    logf = open(log_path, "w")  # noqa: SIM115 - must outlive this cell
    proc = subprocess.Popen(engine_argv(i), stdout=logf, stderr=subprocess.STDOUT, env=env)
    SERVERS[i] = {"proc": proc, "logf": logf, "log_path": str(log_path)}
    print(f"launched replica {i}: GPU {i}, port {PORTS[i]}")


def log_text(i):
    return pathlib.Path(SERVERS[i]["log_path"]).read_text(errors="replace") if i in SERVERS else ""


def http(i, method, path, payload=None, timeout=60):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[i]}{path}", data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")
    except Exception as e:
        return 0, repr(e)


def wait_ready(i, timeout=STARTUP_TIMEOUT_S, every=30):
    """Ready = OUR process is alive AND the port answers, checked in that order."""
    t0, last = time.time(), 0.0
    while time.time() - t0 < timeout:
        if SERVERS[i]["proc"].poll() is not None:
            print(f"  !! replica {i} EXITED (returncode {SERVERS[i]['proc'].returncode}). Log tail:")
            print(log_text(i)[-3000:])
            return False
        if http(i, "GET", "/health", timeout=5)[0] == 200:
            print(f"  replica {i} ready in {time.time() - t0:.0f}s")
            return True
        if time.time() - t0 - last >= every:
            last = time.time() - t0
            tail = log_text(i).strip().splitlines()
            print(f"    ...{last:.0f}s  {tail[-1][:110] if tail else '(log empty)'}")
        time.sleep(5)
    print(f"  !! replica {i} not ready within {timeout}s")
    return False


def stop_server(i):
    if i not in SERVERS:
        return
    SERVERS[i]["proc"].terminate()
    try:
        SERVERS[i]["proc"].wait(timeout=90)
    except subprocess.TimeoutExpired:
        SERVERS[i]["proc"].kill()
    SERVERS[i]["logf"].close()
    del SERVERS[i]


def stop_all():
    for i in list(SERVERS):
        stop_server(i)


def metric(text, family, **labels):
    """Sum a family over matching labels. None = ABSENT, which is not the same as 0."""
    lab = "".join(f'(?=[^}}]*{k}="{v}")' for k, v in labels.items())
    pat = re.compile(r"^" + re.escape(family) + r"(?:_total)?\{" + lab + r"[^}]*\}\s+(\S+)",
                     re.MULTILINE)
    vals = [float(x) for x in pat.findall(text)]
    return sum(vals) if vals else None


COUNTERS = {  # our name -> (metric family, labels)
    "prefix_cache_queries": ("vllm:prefix_cache_queries", {}),
    "prefix_cache_hits": ("vllm:prefix_cache_hits", {}),
    "local_compute": ("vllm:prompt_tokens_by_source", {"source": "local_compute"}),
    "local_cache_hit": ("vllm:prompt_tokens_by_source", {"source": "local_cache_hit"}),
    "preemptions": ("vllm:num_preemptions", {}),
    "prompt_tokens": ("vllm:prompt_tokens", {}),
    "generation_tokens": ("vllm:generation_tokens", {}),
}


def read_counters(text):
    return {k: metric(text, fam, **lab) for k, (fam, lab) in COUNTERS.items()}


def settled_scrape(i, max_wait=300):
    """Scrape once nothing runs or waits AND the counters stopped moving: stats reach /metrics a
    moment after the tokens they describe, so an early read under-counts."""
    t0, prev = time.time(), None
    while True:
        text = http(i, "GET", "/metrics", timeout=30)[1]
        busy = ((metric(text, "vllm:num_requests_running") or 0)
                + (metric(text, "vllm:num_requests_waiting") or 0))
        now = read_counters(text)
        if busy == 0 and now == prev:
            return text
        if time.time() - t0 > max_wait:
            raise Stop(f"⛔ replica {i} never settled within {max_wait}s (busy={busy})")
        prev = now
        time.sleep(1.0)


def deltas(before, after):
    b, a = read_counters(before), read_counters(after)
    return {k: None if a[k] is None else round(a[k] - (b[k] or 0.0)) for k in COUNTERS}


def fresh_caches():
    """A cold cache on BOTH replicas before every run, by /reset_prefix_cache (a restart if the
    endpoint is not served)."""
    global RESET_MODE
    for i in range(N_REPLICAS):
        if RESET_MODE == "endpoint":
            for _ in range(15):
                code, body = http(i, "POST", "/reset_prefix_cache")
                try:
                    ok = code == 200 and json.loads(body).get("success") is True
                except ValueError:
                    ok = False
                if ok:
                    break
                if code in (404, 405):
                    print("  /reset_prefix_cache is not served: restarting the engines per run")
                    RESET_MODE = "restart"
                    break
                time.sleep(2)  # success=false: a finishing request still holds blocks
            else:
                raise Stop(f"⛔ replica {i} refused to reset its prefix cache 15 times")
        if RESET_MODE == "restart":
            stop_server(i)
            start_server(i)
            if not wait_ready(i):
                raise Stop(f"⛔ replica {i} failed to restart — the log tail is above")


print("server helpers ready")


In [ ]:
CAPACITY = {}
for _i in range(N_REPLICAS):  # one at a time: the first start downloads the weights for both
    start_server(_i)
    if not wait_ready(_i):
        raise Stop(f"⛔ replica {_i} did not start — the log tail is above. Download the zip.")
    _log = log_text(_i)
    _m = re.search(r"KV cache size: ([\d,]+) tokens", _log)
    CAPACITY[_i] = {
        "kv_cache_size_tokens_logged": int(_m.group(1).replace(",", "")) if _m else None,
        "expected_tokens": NUM_GPU_BLOCKS * BLOCK_SIZE,
        "override_logged": f"num_gpu_blocks_override={NUM_GPU_BLOCKS}" in _log,
    }
    print(f"  replica {_i}: {CAPACITY[_i]}")
    if CAPACITY[_i]["kv_cache_size_tokens_logged"] != CAPACITY[_i]["expected_tokens"]:
        raise Stop(f"⛔ replica {_i} logged {CAPACITY[_i]['kv_cache_size_tokens_logged']} tokens "
                   f"of KV cache, not {CAPACITY[_i]['expected_tokens']:,}: every prediction "
                   f"assumes exactly {NUM_GPU_BLOCKS} blocks per replica.")


## 6 — The client, the four parts, and saving


In [ ]:
from statistics import median


def stream_one(i, ids, max_tokens):
    """One streamed completion to replica i: send time, the arrival time of EVERY chunk (one chunk
    per token on vLLM 0.30 — checked in the warm-up below), and the usage block."""
    body = {"model": MODEL, "prompt": ids, "max_tokens": max_tokens, "temperature": 0.0,
            "ignore_eos": True, "stream": True, "stream_options": {"include_usage": True}}
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[i]}/v1/completions",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    rec = {"t_send": time.perf_counter(), "t_first": None, "t_chunks": [], "usage": None,
           "finish_reason": None, "error": None}
    try:
        with urllib.request.urlopen(req, timeout=1800) as resp:
            for raw in resp:
                line = raw.strip()
                if not line.startswith(b"data:"):
                    continue
                data = line[5:].strip()
                if data == b"[DONE]":
                    break
                obj = json.loads(data)
                if obj.get("choices"):
                    t = time.perf_counter()
                    rec["t_chunks"].append(t)
                    if rec["t_first"] is None:
                        rec["t_first"] = t
                    reason = obj["choices"][0].get("finish_reason")
                    if reason:
                        rec["finish_reason"] = reason
                if obj.get("usage"):
                    rec["usage"] = obj["usage"]
    except Exception as e:
        rec["error"] = repr(e)
    rec["t_done"] = time.perf_counter()
    return rec


def rand_ids(rng, n):
    """Ordinary text tokens, random: two prompts share no 16-token block by chance."""
    return [rng.randrange(1000, 100_000) for _ in range(n)]


def usable(rec, prompt_len, out):
    u = rec["usage"] or {}
    return (rec["error"] is None and rec["finish_reason"] == "length"
            and u.get("completion_tokens") == out and u.get("prompt_tokens") == prompt_len)


def itl_totals(text):
    return (metric(text, "vllm:inter_token_latency_seconds_sum"),
            metric(text, "vllm:inter_token_latency_seconds_count"))


def itl_mean(before, after):
    """Engine-side mean inter-token latency between two scrapes (vLLM measures it itself)."""
    (s0, c0), (s1, c1) = itl_totals(before), itl_totals(after)
    if None in (s0, c0, s1, c1) or c1 - c0 <= 0:
        return None
    return (s1 - s0) / (c1 - c0)


def reset_replica(i):
    """A cold cache on ONE replica (the other may be busy in part C). Timing needs no reset —
    prompts are random — so a refusal is recorded, never fatal."""
    for _ in range(15):
        code, body = http(i, "POST", "/reset_prefix_cache")
        try:
            if code == 200 and json.loads(body).get("success") is True:
                return True
        except ValueError:
            pass
        if code in (404, 405):
            return False
        time.sleep(2)
    return False


def nearest_rank(sorted_vals, q):
    if not sorted_vals:
        return None
    return sorted_vals[max(0, min(len(sorted_vals) - 1, round(q * (len(sorted_vals) - 1))))]


def measure_decode(i, L, b, seed):
    """b requests of L random tokens at once. A request's decode step = mean gap from token `skip`
    to its last; the cell's = the median over requests. Invalid if any request failed or anything
    was preempted — a preempted request's gaps would time recomputation, not decode."""
    out, skip = DESIGN["decode"]["output_tokens"], DESIGN["decode"]["skip_tokens"]
    rng = random.Random(seed)
    prompts = [rand_ids(rng, L) for _ in range(b)]
    was_reset = reset_replica(i)
    before = settled_scrape(i)
    with ThreadPoolExecutor(max_workers=b) as pool:
        recs = list(pool.map(lambda ids: stream_one(i, ids, out), prompts))
    after = settled_scrape(i)
    d = deltas(before, after)
    steps = []
    for r in recs:
        t = r["t_chunks"]
        if usable(r, L, out) and len(t) == out:
            steps.append((t[-1] - t[skip]) / (out - 1 - skip))
    return {
        "L": L, "b": b, "seed": seed, "reset": was_reset,
        "valid": len(steps) == b and (d["preemptions"] or 0) == 0,
        "step_s": median(steps) if steps else None,
        "request_steps_s": [round(x, 6) for x in steps],
        "ttft_s": [round(r["t_first"] - r["t_send"], 6) for r in recs if r["t_first"]],
        "server_itl_s": itl_mean(before, after),
        "preemptions": d["preemptions"], "failed": b - len(steps),
    }


def measure_prefill(i, c, n, seed):
    """A warm request of c tokens, then c + n tokens with the same first c: TTFT of the second."""
    rng = random.Random(seed)
    prefix, new = rand_ids(rng, c), rand_ids(rng, n)
    reset_replica(i)
    settled_scrape(i)
    if c:
        warm = stream_one(i, prefix, 1)
        if not usable(warm, c, 1):
            return {"c": c, "n": n, "seed": seed, "valid": False, "error": warm["error"]}
        settled_scrape(i)
    rec = stream_one(i, prefix + new, 1)
    u = rec["usage"] or {}
    cached = (u.get("prompt_tokens_details") or {}).get("cached_tokens")
    return {
        "c": c, "n": n, "seed": seed, "cached_tokens": cached, "prompt_tokens": c + n,
        "ttft_s": None if rec["t_first"] is None else round(rec["t_first"] - rec["t_send"], 6),
        "valid": usable(rec, c + n, 1) and cached is not None and rec["t_first"] is not None,
        "error": rec["error"],
    }


# Part C's background load: a SEPARATE process streaming to replica 1, as Phase 6's client did —
# its CPU use is the point. It stops when the stop file appears.
BG_SCRIPT = r"""
import json, random, sys, threading, time, urllib.request, pathlib
port, conc, shared_n, private_n, out, stop, model = sys.argv[1:8]
conc, shared_n, private_n, out = int(conc), int(shared_n), int(private_n), int(out)
shared = [1000 + (7 * j) % 90000 for j in range(shared_n)]
done = [0]
def worker(k):
    rng = random.Random(k)
    while not pathlib.Path(stop).exists():
        ids = shared + [rng.randrange(1000, 100000) for _ in range(private_n)]
        body = {"model": model, "prompt": ids, "max_tokens": out, "temperature": 0.0,
                "ignore_eos": True, "stream": True, "stream_options": {"include_usage": True}}
        req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/completions",
                                     data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=1800) as resp:
                for raw in resp:
                    line = raw.strip()
                    if line.startswith(b"data:") and line[5:].strip() != b"[DONE]":
                        json.loads(line[5:].strip())
            done[0] += 1
        except Exception:
            time.sleep(0.5)
threads = [threading.Thread(target=worker, args=(k,), daemon=True) for k in range(conc)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(json.dumps({"completed": done[0]}))
"""


def start_background():
    bg = DESIGN["cpu"]["background"]
    script = pathlib.Path(OUT_DIR) / "bg_load.py"
    script.write_text(BG_SCRIPT)
    stop = pathlib.Path(OUT_DIR) / "bg.stop"
    stop.unlink(missing_ok=True)
    log = open(pathlib.Path(OUT_DIR) / "bg_load.log", "a")  # noqa: SIM115 - outlives the call
    proc = subprocess.Popen(
        [sys.executable, str(script), str(PORTS[bg["replica"]]), str(bg["concurrency"]),
         str(bg["shared_prefix_tokens"]), str(bg["private_tokens"]), str(bg["output_tokens"]),
         str(stop), MODEL], stdout=log, stderr=subprocess.STDOUT)
    return {"proc": proc, "stop": stop, "log": log}


def stop_background(bg):
    bg["stop"].touch()
    try:
        bg["proc"].wait(timeout=600)
    except subprocess.TimeoutExpired:
        bg["proc"].kill()
    bg["log"].close()
    settled_scrape(DESIGN["cpu"]["background"]["replica"], max_wait=600)


def run_holdout(name):
    """Phase 6's traffic on ONE engine, open-loop at its arrival times."""
    trace, out = TRACES[name], WORKLOADS[name]["output_tokens"]
    reset_replica(MEASURE)
    before = settled_scrape(MEASURE)
    t_zero = time.perf_counter() + 2.0
    with ThreadPoolExecutor(max_workers=len(trace)) as pool:
        futs = []
        for q in trace:
            delay = t_zero + q["arrival_ts"] - time.perf_counter()
            if delay > 0:
                time.sleep(delay)
            futs.append((q, pool.submit(stream_one, MEASURE, q["ids"], out)))
        results = [(q, f.result()) for q, f in futs]
    after = settled_scrape(MEASURE, max_wait=900)
    d = deltas(before, after)
    recs = []
    for q, rec in results:
        u = rec["usage"] or {}
        recs.append({
            "request_id": q["request_id"],
            "valid": usable(rec, len(q["ids"]), out),
            "cached_tokens": (u.get("prompt_tokens_details") or {}).get("cached_tokens"),
            "ttft_ms": None if rec["t_first"] is None else (rec["t_first"] - rec["t_send"]) * 1e3,
            "send_offset_s": rec["t_send"] - t_zero, "scheduled_offset_s": q["arrival_ts"],
        })
    ttfts = sorted(r["ttft_ms"] for r in recs if r["ttft_ms"] is not None)
    hits, queries = d["prefix_cache_hits"] or 0, d["prefix_cache_queries"] or 0
    run = {
        "workload": name, "counters": d, "hit_rate": hits / queries if queries else None,
        "preemptions": d["preemptions"], "server_itl_s": itl_mean(before, after),
        "ttft_p50_ms": nearest_rank(ttfts, 0.50), "ttft_p95_ms": nearest_rank(ttfts, 0.95),
        "invalid": sum(1 for r in recs if not r["valid"]),
        "worst_send_lag_ms": max(r["send_offset_s"] - r["scheduled_offset_s"] for r in recs) * 1e3,
        "requests": recs,
    }
    pred = PRED["runs"][name]["context-free profile"]
    print(f"  {name:12s} hit {run['hit_rate']:.4f} (context-free model {pred['hit_rate']:.4f}) · "
          f"p50 {run['ttft_p50_ms'] or 0:.0f} ms, p95 {run['ttft_p95_ms'] or 0:.0f} ms (context-free "
          f"model {pred['ttft_p50_ms']:.0f} / {pred['ttft_p95_ms']:.0f}) · preemptions "
          f"{run['preemptions']} · {run['invalid']} invalid · send lag ≤ {run['worst_send_lag_ms']:.0f} ms")
    return run


RESULTS = {"notebook": "notebooks/c2_timing_t4.ipynb", "design": DESIGN,
           "fingerprints": FINGERPRINTS, "predictions_sha256": PREDICTIONS_SHA256}


def save():
    """Write everything measured so far, and the zip: a later failure keeps the earlier parts."""
    RESULTS.update({"environment": ENV, "metric_and_source_audit": AUDIT, "capacity": CAPACITY,
                    "engine_argv": [engine_argv(i) for i in range(N_REPLICAS)]})
    pathlib.Path(OUT_DIR, "c2.json").write_text(json.dumps(RESULTS, indent=1, default=str))
    return shutil.make_archive(OUT_DIR, "zip", OUT_DIR)


print("client, parts and saving ready")


In [ ]:
# Throwaway requests on ids that share nothing with the traces (every run below starts from a reset)
# — and a check, before any measured run, that each response reports what the comparison needs.
for _i in range(N_REPLICAS):
    _ids = [1000 + 7 * j + _i for j in range(200)]
    _rec = stream_one(_i, _ids, 8)
    _u = _rec["usage"] or {}
    if (_rec["error"] or _rec["finish_reason"] != "length" or _u.get("completion_tokens") != 8
            or (_u.get("prompt_tokens_details") or {}).get("cached_tokens") is None):
        raise Stop(f"⛔ replica {_i}'s warm-up response is unusable — is ignore_eos honoured, is "
                   f"--enable-prompt-tokens-details in effect? {_rec}")
    _absent = [k for k, v in read_counters(http(_i, "GET", "/metrics")[1]).items() if v is None]
    if _absent:
        raise Stop(f"⛔ replica {_i}'s /metrics lacks counters the comparison reads: {_absent}")
print("warm-up done: both replicas report cached_tokens, exact lengths and every counter")


In [ ]:
# The design reads decode steps from chunk arrival times: that needs one chunk per token.
_rec = stream_one(MEASURE, [1000 + j for j in range(64)], 16)
if not usable(_rec, 64, 16) or len(_rec["t_chunks"]) != 16:
    raise Stop(f"⛔ 16 tokens arrived in {len(_rec['t_chunks'])} chunks: part A's step timing "
               "assumes one chunk per token")
_itl = itl_totals(http(MEASURE, "GET", "/metrics")[1])
if None in _itl:
    print("⚠️  vllm:inter_token_latency_seconds is absent: part C's engine-side test cannot be "
          "decided (the client-side steps are still recorded)")
print(f"one chunk per token ✓ · engine-side inter-token latency {'✓' if None not in _itl else '—'}")


## 7 — Part A: decode step against context length × batch size (replica 1 idle)


In [ ]:
DECODE = []
_d = DESIGN["decode"]
for _rep in range(_d["reps"]):
    for _L, _b in _d["grid"]:
        _m = measure_decode(MEASURE, _L, _b, seed=f"A-{_L}-{_b}-{_rep}")
        DECODE.append({"rep": _rep, **_m})
        _itl = _m["server_itl_s"]
        print(f"  A rep {_rep}  L {_L:>5}  b {_b:>3}  step {(_m['step_s'] or 0) * 1e3:7.1f} ms  "
              f"engine ITL {(_itl or 0) * 1e3:7.1f} ms  preemptions {_m['preemptions']}  "
              f"{'✓' if _m['valid'] else '⛔ invalid'}")
RESULTS["decode"] = DECODE
print(f"saved {save()}")


## 8 — Part B: prefill with and without a cached prefix (replica 1 idle)


In [ ]:
PREFILL = []
_p = DESIGN["prefill"]
for _rep in range(_p["reps"]):
    for _c, _n in _p["grid"]:
        _m = measure_prefill(MEASURE, _c, _n, seed=f"B-{_c}-{_n}-{_rep}")
        PREFILL.append({"rep": _rep, **_m})
        print(f"  B rep {_rep}  cached {_c:>5} (engine: {_m.get('cached_tokens')})  new {_n:>5}  "
              f"TTFT {(_m.get('ttft_s') or 0) * 1e3:7.1f} ms  {'✓' if _m['valid'] else '⛔ invalid'}")
RESULTS["prefill"] = PREFILL
print(f"saved {save()}")


## 9 — Part C: CPU contention — the same cells alone and with replica 1 busy


In [ ]:
CPU = []
_c = DESIGN["cpu"]
for _rep in range(_c["reps"]):
    for _cond in ("alone", "contended"):
        _bg = None
        if _cond == "contended":
            _bg = start_background()
            time.sleep(_c["settle_s"])
        try:
            for _L, _b in _c["cells"]:
                _m = measure_decode(MEASURE, _L, _b, seed=f"C-{_L}-{_b}-{_rep}-{_cond}")
                CPU.append({"rep": _rep, "condition": _cond, **_m})
                print(f"  C rep {_rep}  {_cond:9s}  L {_L:>5}  b {_b:>3}  engine ITL "
                      f"{(_m['server_itl_s'] or 0) * 1e3:7.1f} ms  client step "
                      f"{(_m['step_s'] or 0) * 1e3:7.1f} ms  {'✓' if _m['valid'] else '⛔ invalid'}")
        finally:
            if _bg is not None:
                stop_background(_bg)
RESULTS["cpu"] = CPU
print(f"saved {save()}")


## 10 — Part D: held-out traffic on one engine


In [ ]:
HOLDOUT = []
for _name in WORKLOADS:
    HOLDOUT.append(run_holdout(_name))
RESULTS["holdout"] = HOLDOUT
_zip = save()
print(f"\nsaved {_zip} ({pathlib.Path(_zip).stat().st_size / 1e6:.1f} MB)")
print("⭐ Download c2_results.zip and bring it back — then run the clean-up cell.")


## 11 — Clean-up (run after downloading the zip)


In [ ]:
stop_all()
print(sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv"))
